# 03 – Baseline Models: ARIMA, Prophet, XGBoost

This notebook trains and evaluates three baseline forecasting models:
- **ARIMA** – classical statistical time-series model
- **Prophet** – Facebook/Meta time-series model with trend/seasonality decomposition
- **XGBoost** – gradient-boosted trees with engineered features

Predictions are saved for use in notebook 05 (ensemble).

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plotter = Plotter()

In [2]:
# ── Load processed data ─────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

target_col = 'Close'
y_test = test[target_col]

Train: 1006 | Val: 252 | Test: 249


## A. ARIMA

In [ ]:
# ── ARIMA Model ──────────────────────────────────────────────────────────
from src.models.arima_model import ARIMAModel
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter

plotter = Plotter()
arima_order = (2, 1, 0)
window = 252
arima = ARIMAModel(order=arima_order)



print(f"--- Training ARIMA (Walk-forward | pdq={arima_order} | window={window} | Metrics=Validation) ---")

arima_val_preds, arima_test_preds = arima.train_and_refit(
train, val, test, target_col="Close", window=window
)

print("\n [Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val['Close'], arima_val_preds))

print("\n [Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test['Close'], arima_test_preds))

# 参数标签：用于 title 和 filename
p, d, q = arima_order
pdq_tag = f"p{p}_d{d}_q{q}"
window_tag = f"w{window}"

# 2024 Validation 预测对比图
plotter.plot_predictions_comparison(
    y_true=val["Close"],
    predictions={"ARIMA Forecast": arima_val_preds},
    dates=val.index,
    title=f"ARIMA Validation Predictions (2024 | pdq={arima_order} | window={window})",
    filename=f"arima_val_2024_{pdq_tag}_{window_tag}.png"
)

# 2025 Test 预测对比图
plotter.plot_predictions_comparison(
    y_true=test["Close"],
    predictions={"ARIMA Forecast": arima_test_preds},
    dates=test.index,
    title=f"ARIMA Test Predictions (2025 | pdq={arima_order} | window={window})",
    filename=f"arima_test_2025_{pdq_tag}_{window_tag}.png"
)

## B. Prophet

In [ ]:
# ── 5. Prophet Model ────────────────────────────────────────────────────────
from src.models.prophet_model import ProphetModel

prophet = ProphetModel()
print("--- Training Prophet (Two-Phase) ---")
prophet_val_preds, prophet_test_preds = prophet.train_and_refit(train, val, test)

print("\n [Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val['Close'], prophet_val_preds))

print("\n [Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test['Close'], prophet_test_preds))

plotter.plot_predictions_comparison(
y_true=val["Close"],
predictions={"Prophet Forecast": prophet_val_preds},
dates=val.index,
title="Prophet 2024 Validation Predictions",
filename="prophet_val_forecast_2024.png"
)

plotter.plot_predictions_comparison(
    y_true=test['Close'],
    predictions={'Prophet Forecast': prophet_test_preds},
    dates=test.index,
    title="Prophet 2025 Test Predictions",
    filename="prophet_test_forecast_2025.png"
)

## C. XGBoost

In [3]:
# ── 6. XGBoost Model (Strict Time-Stepped Architecture) ────────────────────
from src.models.xgboost_model import XGBoostModel
import pandas as pd
import matplotlib.pyplot as plt

target_col = 'Close'
# 1. 抓取所有特征，但依然剔除当前行的目标变量（防同日作弊）
base_features = [col for col in train.columns if col not in [target_col, 'Adj Close']]

print("Engineering Lag-1 features to predict Tomorrow using Today's data...")

# 2. 
# 把当天的 'Close' 也加入基础特征池中
base_features.append('Close')

# 3. 为 Train, Val, Test 创建专门的 XGBoost 数据集（将所有特征强行推迟1天）
X_train_xgb = train[base_features].shift(1)
X_val_xgb = val[base_features].shift(1)
X_test_xgb = test[base_features].shift(1)

# 由于推迟1天，第一行会变成 NaN，我们用向后填充（bfill）兜底
X_train_xgb.bfill(inplace=True)
X_val_xgb.bfill(inplace=True)
X_test_xgb.bfill(inplace=True)

# 为了防止特征重名混淆，给这些列统一打上 _lag1 的标签
xgb_feature_cols = [f"{col}_lag1" for col in base_features]
X_train_xgb.columns = xgb_feature_cols
X_val_xgb.columns = xgb_feature_cols
X_test_xgb.columns = xgb_feature_cols

# 把原始的目标变量 y 拼回去
train_xgb_df = pd.concat([X_train_xgb, train[target_col]], axis=1)
val_xgb_df = pd.concat([X_val_xgb, val[target_col]], axis=1)
test_xgb_df = pd.concat([X_test_xgb, test[target_col]], axis=1)

print(f"XGBoost is now safely learning from {len(xgb_feature_cols)} past-day features (including 'Close_lag1')!")

# 4. 开始两阶段训练
xgb = XGBoostModel()
print("--- Training XGBoost (Two-Phase Refitting) ---")
# 调用 train_and_refit，传入专属滞后数据集
xgb_val_preds, xgb_test_preds = xgb.train_and_refit(
    train_xgb_df, val_xgb_df, test_xgb_df,
    features=xgb_feature_cols,
    target_col=target_col
)

print("\n[Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val['Close'], xgb_val_preds))

print("\n[Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test['Close'], xgb_test_preds))

plotter.plot_predictions_comparison(
    y_true=val['Close'],
    predictions={'XGBoost Forecast': xgb_val_preds},
    dates=val.index,
    title="XGBoost 2024 Validation Predictions",
    filename="xgboost_val_forecast_2024.png"
)

plotter.plot_predictions_comparison(
    y_true=test['Close'],
    predictions={'XGBoost Forecast': xgb_test_preds},
    dates=test.index,
    title="XGBoost 2025 Test Predictions",
    filename="xgboost_test_forecast_2025.png"
)

# ---- Phase 1-only model for feature importance (train) ----
xgb_phase1 = XGBoostModel()
xgb_phase1.fit(
    train_xgb_df[xgb_feature_cols],
    train_xgb_df[target_col],
    X_val=val_xgb_df[xgb_feature_cols],
    y_val=val_xgb_df[target_col],
)

xgb_importance_train = pd.Series(
    xgb_phase1._model.feature_importances_,
    index=xgb_feature_cols
)

plotter.plot_feature_importance(
    xgb_importance_train,
    top_n=20,
    title="XGBoost Feature Importance (train)",
    filename="xgboost_feature_importance_train.png",
    annotate=True,
    decimals=4,
)

# ---- Phase 2 refitted importance ----
xgb_importance_refit = pd.Series(
    xgb._model.feature_importances_,
    index=xgb_feature_cols
)

plotter.plot_feature_importance(
    xgb_importance_refit,
    top_n=20,
    title="XGBoost Feature Importance (Refitted)",
    filename="xgboost_feature_importance.png",
    annotate=True,
    decimals=4,
)

Engineering Lag-1 features to predict Tomorrow using Today's data...
XGBoost is now safely learning from 31 past-day features (including 'Close_lag1')!
--- Training XGBoost (Two-Phase Refitting) ---

[Phase 1] 2024 Validation Metrics:
{'mse': 8484427.937519943, 'rmse': 2912.804136484282, 'mae': 2604.7217959449404, 'mape': 13.227176690598311, 'directional_accuracy': 0.4900398406374502}

[Phase 2] 2025 Test Metrics:
{'mse': 12934637.57975386, 'rmse': 3596.475716552784, 'mae': 3027.7913685993976, 'mape': 12.731561357518121, 'directional_accuracy': 0.5080645161290323}


## D. Comparison

In [ ]:
# ── 7. Aggregate and Save Final Test Metrics ─────────────────────────────
from src.utils.metrics import calculate_metrics
import pandas as pd
from src.config import RESULTS_DIR

# 重新计算各模型在 Phase 2 (2025 测试集) 上的最终表现
arima_metrics = calculate_metrics(test['Close'], arima_test_preds)
prophet_metrics = calculate_metrics(test['Close'], prophet_test_preds)
xgb_metrics = calculate_metrics(test['Close'], xgb_test_preds)

all_metrics = {
    'ARIMA': arima_metrics,
    'Prophet': prophet_metrics,
    'XGBoost': xgb_metrics,
}

# 转换为 DataFrame 方便展示和保存 (转置 T 是为了让模型名在行，指标在列)
metrics_df = pd.DataFrame(all_metrics).T

print("\n🏆 FINAL BASELINE METRICS (2025 Test Set - Refitted) 🏆")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

# 保存最终的指标表格
metrics_df.to_csv(RESULTS_DIR / 'baseline_metrics.csv')
print(f"\n✅ Metrics successfully saved to {RESULTS_DIR / 'baseline_metrics.csv'}")

# 可选：绘制所有 Baseline 模型的指标对比柱状图
plotter.plot_metrics_comparison(all_metrics, filename='baseline_metrics_comparison.png')

In [ ]:
# ── 8. Save Phase 1 & Phase 2 Predictions ────────────────────────────────
from src.config import RESULTS_DIR
import pandas as pd

# 1. 横向拼接各个模型的结果
val_preds_df = pd.concat([arima_val_preds, prophet_val_preds, xgb_val_preds], axis=1)
test_preds_df = pd.concat([arima_test_preds, prophet_test_preds, xgb_test_preds], axis=1)

# 2. 纵向拼接 2024(Val) 和 2025(Test) 的结果，合并为你原来熟悉的 preds_df
preds_df = pd.concat([val_preds_df, test_preds_df], axis=0)
# 重命名列，保持规范
preds_df.columns = ['ARIMA', 'Prophet', 'XGBoost']

# 3. 完美还原你原来的保存逻辑和打印语句
save_path = RESULTS_DIR / 'baseline_predictions.csv'
preds_df.to_csv(save_path)

print(f"Predictions successfully saved to {save_path}")
print(f"Total rows saved: {len(preds_df)} (Val: {len(val_preds_df)} + Test: {len(test_preds_df)})")

## Summary

See `reports/results/baseline_metrics.csv` for a full metrics table.

Continue to **04_model_lstm.ipynb** for the deep learning model.